## Objective ##
Read the original Parquet file, preserve all 3,475,226 records, add basic source metadata, and publish a managed Delta table.

In [0]:
from pyspark.sql import functions as f

source_path = "/Volumes/workspace/default/raw_landing/yellow_tripdata_2025-01.parquet"
# identify the catalog from volume path
catalog = source_path.split("/")[2]

#createm bronze schema if not exists
spark.sql(f"create schema if not exists `{catalog}`.bronze")

# load data
raw_df= spark.read.parquet(source_path)

bronze_df= (
    raw_df
    .withColumn("_source_file", f.lit("yellow_tripdata_2025-01.parquet"))
    .withColumn("_source_month", f.lit("2025-01"))
    .withColumn("_ingested_at", f.current_timestamp())
)
print ("Bronze Source Rows: ", bronze_df.count())

In [0]:
table_name= f"`{catalog}`.bronze.yellow_taxi_trips"
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)
print("Bronze Table created: ",table_name)

#verify result
saved_df=spark.table(table_name)
print("Saved rows: ", saved_df.count())
print("Saved Columns: ", len(saved_df.columns))

In [0]:
table_name= "workspace.bronze.yellow_taxi_trips"

#inspect table delta metadata
details = spark.sql(f"Describe detail {table_name}")
display(
    details.select("name","format","numFiles","sizeInBytes")
)

# Check saved records & source columns
saved_df = spark.table(table_name)

assert saved_df.count() == raw_df.count()
assert all(col in saved_df.columns for col in raw_df.columns)
assert len(saved_df.columns) == 23

print("PASS: Row counts match")
print("PASS: All original columns are present")
print("PASS: 23 columns stored")
